# Sample current clips

This notebook reads the clips that are currently available under `data/clips/`, joins label metadata from `data/labels.jsonl` when present, and samples 100 clips for inspection.


In [ ]:
import shutil
from pathlib import Path

import pandas as pd
from IPython.display import Video, display

from aitraf.data_ops.schema import LabelsSchema
from aitraf.data_ops.utils import strip_clips_prefix

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
CLIPS_DIR = DATA_DIR / "clips"
SAMPLED_CLIPS_DIR = DATA_DIR / "clips_sampled"
LABELS_PATH = DATA_DIR / "labels.jsonl"

SAMPLE_SIZE = 100
RANDOM_SEED = 7
PREVIEW_COUNT = 5

pd.set_option("display.max_colwidth", 120)

In [ ]:
def s3_uri_to_clip_name(video_uri: str) -> str:
    key = video_uri.replace("s3://aitraf/", "", 1)
    return strip_clips_prefix(Path(key)).name


clip_paths = sorted(CLIPS_DIR.rglob("*.mp4"))
if not clip_paths:
    raise FileNotFoundError(f"No clips found under {CLIPS_DIR}")

clips_df = pd.DataFrame({"clip_path": clip_paths})
clips_df["clip_name"] = clips_df["clip_path"].map(lambda path: path.name)
clips_df["clip_relpath"] = clips_df["clip_path"].map(
    lambda path: path.relative_to(PROJECT_ROOT).as_posix()
)

if LABELS_PATH.exists():
    labels_df = pd.read_json(LABELS_PATH, lines=True, dtype=LabelsSchema.types)
    labels_df["clip_name"] = labels_df["video"].map(s3_uri_to_clip_name)
    labels_df = labels_df[
        [
            "clip_name",
            "video",
            "trick",
            "person",
            "key_foot",
            "execution_score",
        ]
    ]
    current_clips_df = clips_df.merge(labels_df, on="clip_name", how="left")
else:
    current_clips_df = clips_df.assign(
        video=pd.NA,
        trick=pd.NA,
        person=pd.NA,
        key_foot=pd.NA,
        execution_score=pd.NA,
    )

unlabeled_clips_df = current_clips_df[current_clips_df["trick"].isna()].copy()

deleted_count = 0
for clip_path in unlabeled_clips_df["clip_path"]:
    if clip_path.exists():
        clip_path.unlink()
        deleted_count += 1
current_clips_df = current_clips_df[current_clips_df["trick"].notna()].reset_index(
    drop=True
)
print(f"Deleted {deleted_count} unlabeled clips from {CLIPS_DIR}")

sample_n = min(SAMPLE_SIZE, len(current_clips_df))
sampled_df = (
    current_clips_df.sample(n=sample_n, random_state=RANDOM_SEED)
    .sort_values(["trick", "person", "clip_name"], na_position="last")
    .reset_index(drop=True)
)

matched_labels = int(current_clips_df["trick"].notna().sum())
copied_count = 0
sampled_copy_paths = []
for row in sampled_df.itertuples(index=False):
    destination_path = SAMPLED_CLIPS_DIR / row.clip_path.relative_to(CLIPS_DIR)
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(row.clip_path, destination_path)
    sampled_copy_paths.append(destination_path)
    copied_count += 1

sampled_df["sampled_clip_path"] = sampled_copy_paths
sampled_df["sampled_clip_relpath"] = sampled_df["sampled_clip_path"].map(
    lambda path: path.relative_to(PROJECT_ROOT).as_posix()
)

print(f"Current clips found: {len(current_clips_df)}")
print(f"Clips matched with labels: {matched_labels}")
print(f"Sampling {sample_n} clips with random seed {RANDOM_SEED}")
print(f"Copied {copied_count} sampled clips to {SAMPLED_CLIPS_DIR}")

sampled_df.head()

In [ ]:
summary_df = pd.DataFrame(
    [
        {"metric": "current_clips", "value": len(current_clips_df)},
        {
            "metric": "clips_with_labels",
            "value": int(current_clips_df["trick"].notna().sum()),
        },
        {
            "metric": "clips_without_labels",
            "value": 0,
        },
        {"metric": "deleted_unlabeled_clips", "value": deleted_count},
        {"metric": "copied_sampled_clips", "value": copied_count},
        {"metric": "sample_size", "value": len(sampled_df)},
    ]
)

sample_trick_counts = (
    sampled_df["trick"]
    .fillna("unlabeled")
    .value_counts()
    .rename_axis("trick")
    .to_frame("sample_count")
)

display(summary_df)
display(sample_trick_counts)
display(
    sampled_df[
        [
            "clip_name",
            "clip_relpath",
            "sampled_clip_relpath",
            "trick",
            "person",
            "key_foot",
            "execution_score",
        ]
    ]
)

In [ ]:
# Preview a few of the sampled clips inline.
preview_df = sampled_df.head(PREVIEW_COUNT)
display(preview_df[["clip_name", "trick", "person", "execution_score"]])

for row in preview_df.itertuples(index=False):
    print(
        f"{row.clip_name} | trick={row.trick} | person={row.person} | score={row.execution_score}"
    )
    display(Video(str(row.clip_path), embed=False, width=320))